In [1]:
import pandas as pd

from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\capela"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\capela\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\capela\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file1 = os.path.join(dwg_folder, "PV_Capela - Mesas 16-32.dwg")
dwg_file2 = os.path.join(dwg_folder, "PV Capela - Eletromecânico + DC.dwg")

shutil.copy(dwg_file1, temp_folder)
shutil.copy(dwg_file2, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file1 = os.path.join(output_folder, "PV_Capela - Mesas 16-32.dxf")
dxf_file2 = os.path.join(output_folder, "PV Capela - Eletromecânico + DC.dxf")

temp_dwg_file1 = os.path.join(temp_folder, "PV_Capela - Mesas 16-32.dwg")
temp_dwg_file2 = os.path.join(temp_folder, "PV Capela - Eletromecânico + DC.dwg")

data_lines = extract_data_from_dwf(dxf_file1)
print("Data Loaded")
data_ELM = extract_data_from_dwf(dxf_file2)
print("ELM Data Loaded")

os.remove(dxf_file1)
os.remove(dxf_file2)
os.remove(temp_dwg_file1)
os.remove(temp_dwg_file2)

Data Loaded
ELM Data Loaded


First we allocate the 16 and 32 structures according to the lines provided.
=

In [2]:
df_lines = pd.DataFrame(data_lines)
df_lines['middle_x'] = (df_lines['start_x'] + df_lines['end_x'])/2
print(df_lines.describe())
print(df_lines.info())
print(df_lines)

            start_x        start_y         end_x          end_y       length  \
count   8563.000000    8563.000000   8563.000000    8563.000000  8563.000000   
mean  -18657.812596  157476.082846 -18647.142393  157476.085590    10.690650   
std      457.770726     594.876033    457.773426     594.880157     0.892555   
min   -19719.657990  156354.122600 -19708.983990  156354.122600    10.584000   
25%   -19013.940940  156925.622600 -19003.266940  156925.622600    10.674000   
50%   -18675.799290  157633.122600 -18665.125290  157633.122600    10.674000   
75%   -18286.141990  157987.622600 -18275.467990  157987.622600    10.674000   
max   -17796.983340  158366.315229 -17786.309340  158374.065229    58.696000   

           middle_x  
count   8563.000000  
mean  -18652.477494  
std      457.772065  
min   -19714.320990  
25%   -19008.603940  
50%   -18670.462290  
75%   -18280.804990  
max   -17791.646340  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8563 entries, 0 to 8562
Data co

In [3]:
df_lines_16 = df_lines[df_lines['layer'] == "ZAGO-MODULO_16"].copy()
df_lines_16["Script"] = df_lines_16.apply(lambda row: (f'(command-s "._insert" "EMF379_estrutura_16" "{round(row['middle_x'],4)},{round(row['start_y'],4)}" 1 1 0)'), axis=1)
df_lines_32 = df_lines[df_lines['layer'] == "ZAGO-MODULO_32"].copy()
print(df_lines_16)
print(df_lines_32)

      type           layer handle      start_x      start_y        end_x  \
0     LINE  ZAGO-MODULO_16   1020 -19337.96509  156454.6226 -19327.29109   
1     LINE  ZAGO-MODULO_16   1021 -19327.29109  156454.6226 -19316.61709   
2     LINE  ZAGO-MODULO_16   1022 -19316.61709  156454.6226 -19305.94309   
3     LINE  ZAGO-MODULO_16   1023 -19305.94309  156454.6226 -19295.26909   
4     LINE  ZAGO-MODULO_16   1024 -19295.26909  156454.6226 -19284.59509   
...    ...             ...    ...          ...          ...          ...   
5115  LINE  ZAGO-MODULO_16   2741 -19085.82891  156354.1226 -19075.15491   
5116  LINE  ZAGO-MODULO_16   2742 -19060.60594  156457.6226 -19049.93194   
5117  LINE  ZAGO-MODULO_16   2743 -19049.93194  156457.6226 -19039.25794   
5118  LINE  ZAGO-MODULO_16   2744 -19039.25794  156457.6226 -19028.58394   
5119  LINE  ZAGO-MODULO_16   2745 -19028.58394  156457.6226 -19017.90994   

            end_y  length     middle_x  \
0     156454.6226  10.674 -19332.62809   
1  

In [4]:
def group_structures(df, max_distance=15):
    """
    Groups adjacent lines in pairs (2) while respecting:
    - Maximum distance constraint (15m default)
    - Sorted by Position Y then X
    """
    # Sort by Position X (primary) and Position Y (secondary)

    df = df.sort_values(['start_y', 'middle_x'], ascending=[False, True]).reset_index(drop=True)



    # Calculate pairwise distances between consecutive structures
    df['x_diff'] = df['middle_x'].diff()
    df['y_diff'] = df['start_y'].diff()
    df['distance'] = np.sqrt(df['x_diff']**2 + df['y_diff']**2)

    # Create group boundaries where:
    # 1. TAG changes OR #### THIS IS NO LONGER A BOUNDARY ####
    # 2. Distance > max_distance OR
    # 3. It's the first structure
    df['group_boundary'] = (
        #(df['TAG'] != df['TAG'].shift(1)) |
        (df['distance'] > max_distance) |
        (df.index == 0)
    )
    # Create group IDs
    df['group_id'] = df['group_boundary'].cumsum()

    # Split into groups
    result_groups = []

    for _, group_df in df.groupby('group_id'):
        size = len(group_df)
        if size <= 2:
            result_groups.append(group_df)

            continue

        n_pairs = size // 2
        for i in range(n_pairs):
            result_groups.append(group_df.iloc[i*2:i*2+2])


        if size % 2:  # Handle odd-sized groups
            result_groups.append(group_df.iloc[-1:])

    return result_groups

def create_group_positions(groups):
    """Create the output DataFrame from groups (same as before)"""
    group_data = []
    for i, group_df in enumerate(groups, 1):
        first_line = group_df.iloc[0]
        group_data.append({
            'Group_ID': i,
            'Position_X': first_line['end_x'],
            'Position_Y': first_line['end_y'],
            'Group_Size': len(group_df),
        })
    return pd.DataFrame(group_data)

In [5]:
results_32 = group_structures(df_lines_32)
group_positions = create_group_positions(results_32)
group_positions["Script"] = group_positions.apply(lambda row: (f'(command-s "._insert" "EMF379_estrutura_32" "{round(row['Position_X'],4)},{round(row['Position_Y'],4)}" 1 1 0)'), axis=1)
print(group_positions)

      Group_ID   Position_X   Position_Y  Group_Size  \
0            1 -18328.29799  158304.6226           2   
1            2 -18296.27599  158304.6226           2   
2            3 -18264.25399  158304.6226           2   
3            4 -18363.05599  158296.6226           2   
4            5 -18277.66399  158296.6226           2   
...        ...          ...          ...         ...   
1715      1716 -19117.85091  156395.1226           2   
1716      1717 -19096.50291  156395.1226           2   
1717      1718 -19128.52491  156387.1226           2   
1718      1719 -19107.17691  156371.1226           2   
1719      1720 -19085.82891  156371.1226           2   

                                                 Script  
0     (command-s "._insert" "EMF379_estrutura_32" "-...  
1     (command-s "._insert" "EMF379_estrutura_32" "-...  
2     (command-s "._insert" "EMF379_estrutura_32" "-...  
3     (command-s "._insert" "EMF379_estrutura_32" "-...  
4     (command-s "._insert" "EMF379_e

In [6]:
with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Bridges\Estruturas-16-32.scr", 'w') as f:
    f.write('\n'.join(group_positions["Script"].astype(str)) + '\n')
    f.write('\n'.join(df_lines_16['Script'].astype(str)) + '\n')

Then we compare the allocated structures (mechanical allocation) to the structures we had before (electrical allocation) to see how many strings were separated.
=

In [7]:
data_ELM_df = pd.DataFrame(data_ELM)
data_ELM_df.drop(columns=["type", "handle", "rotation"], inplace=True)

mask = (data_ELM_df['name'] == "EMF379_string_1") | (data_ELM_df['name'] == "EMF379_string_2")

strings_df = data_ELM_df[mask].copy()
print(strings_df.describe())
print(strings_df.info())
print(strings_df)

            start_x        start_y  end_x  end_y  length
count   4280.000000    4280.000000    0.0    0.0     0.0
mean  -18652.351424  157475.770896    NaN    NaN     NaN
std      457.690980     594.782582    NaN    NaN     NaN
min   -19698.300000  156352.961300    NaN    NaN     NaN
25%   -19008.387100  156923.041950    NaN    NaN     NaN
50%   -18667.613500  157632.541950    NaN    NaN     NaN
75%   -18280.373350  157986.783900    NaN    NaN     NaN
max   -17796.983300  158305.783900    NaN    NaN     NaN
<class 'pandas.core.frame.DataFrame'>
Index: 4280 entries, 43 to 7072
Data columns (total 17 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   layer    4280 non-null   object 
 1   name     4280 non-null   object 
 2   start_x  4280 non-null   float64
 3   start_y  4280 non-null   float64
 4   PRK      4280 non-null   object 
 5   UFV      4280 non-null   object 
 6   PST      4280 non-null   object 
 7   INV      4280 non-null   object 
 8  

In [8]:
strings_df.rename(columns={"start_x": "middle_x", "start_y": "middle_y"}, inplace=True)

for idx, row in strings_df.iterrows():
    x = row['middle_x']
    y = row['middle_y']

    mask = (
        (strings_df['middle_y'] >= y - 5) &
        (strings_df['middle_y'] <= y + 5) &
        (strings_df['middle_x'] >= x - 0.5) &
        (strings_df['middle_x'] <= x + 0.5) &
        (strings_df.index != idx)
    )

    if mask.any():
        y_neighbors = strings_df.loc[mask, 'middle_y']

        if (y > y_neighbors).any():
            strings_df.loc[idx, 'Type'] = "T"
        else:
            strings_df.loc[idx, 'Type'] = "B"

    else:
        strings_df.loc[idx, 'Type'] = "C"


strings_df.loc[:,'Line'] = np.where(
    strings_df["Type"] == "C",
    round(strings_df["middle_y"]),
    np.where(
        strings_df["Type"] == "T",
        round(strings_df["middle_y"] - 1.1613),
        round(strings_df["middle_y"] + 1.1613)
    )
)

strings_df['start_x'] = np.where(
    strings_df["Type"] == "C",
    strings_df['middle_x'] - 10.574,
    strings_df['middle_x'] - 21.148
)

strings_df['end_x'] = np.where(
    strings_df["Type"] == "C",
    strings_df['middle_x'] + 10.574,
    strings_df['middle_x'] + 21.148
)


strings_df.sort_values(by=["Line", "middle_x"], ascending=[False,True], inplace=True, ignore_index=True)
strings_df["PRK.UFV.PST.INV.STR"] = strings_df["PRK"].astype(str) + "." + strings_df["UFV"].astype(str) + "." + strings_df["PST"].astype(str) + "." + strings_df["INV"].astype(str) + "." + strings_df["STR"].astype(str)
strings_df.drop(columns=['layer', 'name', 'UFV', 'PST', 'INV', 'QBT', 'DSJ', 'POT', 'GRP', 'EFX', 'STR', 'end_y', 'length'], inplace=True)
print(strings_df)

        middle_x     middle_y PRK       end_x Type      Line     start_x  \
0    -18349.6460  158304.6226   2 -18339.0720    C  158305.0 -18360.2200   
1    -18317.6140  158305.7839   2 -18296.4660    T  158305.0 -18338.7620   
2    -18317.6140  158303.4613   2 -18296.4660    B  158305.0 -18338.7620   
3    -18274.9180  158305.7839   2 -18253.7700    T  158305.0 -18296.0660   
4    -18274.9180  158303.4613   2 -18253.7700    B  158305.0 -18296.0660   
...          ...          ...  ..         ...  ...       ...         ...   
4275 -19107.1669  156363.2839   1 -19086.0189    T  156362.0 -19128.3149   
4276 -19107.1669  156360.9613   1 -19086.0189    B  156362.0 -19128.3149   
4277 -19075.1549  156362.1226   1 -19064.5809    C  156362.0 -19085.7289   
4278 -19096.4929  156355.2839   1 -19075.3449    T  156354.0 -19117.6409   
4279 -19096.4929  156352.9613   1 -19075.3449    B  156354.0 -19117.6409   

     PRK.UFV.PST.INV.STR  
0             2.1.2.1.01  
1             2.1.2.1.02  
2     

In [9]:
structures_32 = group_positions.drop(columns=['Group_ID', 'Group_Size', 'Script']).copy()
structures_32['No of modules'] = 32
print(structures_32)

structures_16 = df_lines_16.drop(columns=['type', 'layer', 'handle', 'start_x', 'end_x', 'end_y', 'length', 'Script'])
structures_16.rename(columns={'middle_x': 'Position_X', 'start_y': 'Position_Y'}, inplace=True)
structures_16['No of modules'] = 16
print(structures_16)

       Position_X   Position_Y  No of modules
0    -18328.29799  158304.6226             32
1    -18296.27599  158304.6226             32
2    -18264.25399  158304.6226             32
3    -18363.05599  158296.6226             32
4    -18277.66399  158296.6226             32
...           ...          ...            ...
1715 -19117.85091  156395.1226             32
1716 -19096.50291  156395.1226             32
1717 -19128.52491  156387.1226             32
1718 -19107.17691  156371.1226             32
1719 -19085.82891  156371.1226             32

[1720 rows x 3 columns]
       Position_Y   Position_X  No of modules
0     156454.6226 -19332.62809             16
1     156454.6226 -19321.95409             16
2     156454.6226 -19311.28009             16
3     156454.6226 -19300.60609             16
4     156454.6226 -19289.93209             16
...           ...          ...            ...
5115  156354.1226 -19080.49191             16
5116  156457.6226 -19055.26894             16
5117  156

In [10]:
mechanical_structures = pd.concat([structures_32, structures_16], ignore_index=True)
mechanical_structures['Position_Y'] = round(mechanical_structures['Position_Y'])

mechanical_structures['start_x'] = np.where(
    mechanical_structures['No of modules'] == 32,
    mechanical_structures['Position_X'] - 10.574,
    mechanical_structures['Position_X'] - 5.282
)

mechanical_structures['end_x'] = np.where(
    mechanical_structures['No of modules'] == 32,
    mechanical_structures['Position_X'] + 10.574,
    mechanical_structures['Position_X'] + 5.282
)

mechanical_structures.sort_values(by=['Position_Y', 'Position_X'], ascending=[False, True], inplace=True)
print(mechanical_structures)
mechanical_structures.to_excel(r"C:\Users\Usuario\Desktop\mechanical_structures.xlsx")


       Position_X  Position_Y  No of modules      start_x        end_x
5736 -18354.98299    158305.0             16 -18360.26499 -18349.70099
5737 -18344.30899    158305.0             16 -18349.59099 -18339.02699
0    -18328.29799    158305.0             32 -18338.87199 -18317.72399
5738 -18312.28699    158305.0             16 -18317.56899 -18307.00499
1    -18296.27599    158305.0             32 -18306.84999 -18285.70199
...           ...         ...            ...          ...          ...
6831 -19069.81791    156362.0             16 -19075.09991 -19064.53591
6832 -19112.51391    156354.0             16 -19117.79591 -19107.23191
6833 -19101.83991    156354.0             16 -19107.12191 -19096.55791
6834 -19091.16591    156354.0             16 -19096.44791 -19085.88391
6835 -19080.49191    156354.0             16 -19085.77391 -19075.20991

[6840 rows x 5 columns]


In [11]:
#Different Lines
condition1 = mechanical_structures["Position_Y"] != mechanical_structures["Position_Y"].shift()

#Break in the Structure with no 16 modules on the edges
condition2 = ((abs(mechanical_structures["Position_X"] - mechanical_structures["Position_X"].shift()) > 25) &
              (mechanical_structures["No of modules"] != "16") &
              (mechanical_structures["No of modules"].shift() != "16"))

#Break in the Structure with one 16 modules on the edges
condition3 = ((abs(mechanical_structures["Position_X"] - mechanical_structures["start_x"].shift()) > 20) &
              ((mechanical_structures["No of modules"] == "16") != (mechanical_structures["No of modules"].shift() == "16")))

#Break in the Structure with two 16 modules on the edges
condition4 = ((abs(mechanical_structures["start_x"] - mechanical_structures["start_x"].shift()) > 15) &
              (mechanical_structures["No of modules"] == "16") &
              (mechanical_structures["No of modules"].shift() == "16"))

break_condition = condition1 | condition2 | condition3 | condition4


mechanical_structures['Group'] = break_condition.cumsum()
print(mechanical_structures)

       Position_X  Position_Y  No of modules      start_x        end_x  Group
5736 -18354.98299    158305.0             16 -18360.26499 -18349.70099      1
5737 -18344.30899    158305.0             16 -18349.59099 -18339.02699      1
0    -18328.29799    158305.0             32 -18338.87199 -18317.72399      1
5738 -18312.28699    158305.0             16 -18317.56899 -18307.00499      1
1    -18296.27599    158305.0             32 -18306.84999 -18285.70199      1
...           ...         ...            ...          ...          ...    ...
6831 -19069.81791    156362.0             16 -19075.09991 -19064.53591    687
6832 -19112.51391    156354.0             16 -19117.79591 -19107.23191    688
6833 -19101.83991    156354.0             16 -19107.12191 -19096.55791    688
6834 -19091.16591    156354.0             16 -19096.44791 -19085.88391    688
6835 -19080.49191    156354.0             16 -19085.77391 -19075.20991    688

[6840 rows x 6 columns]


In [12]:
number_of_bridges = mechanical_structures.groupby("Group").size()-1
print(number_of_bridges)
print(number_of_bridges.sum())

Group
1      6
2      1
3      1
4      8
5      1
      ..
684    8
685    7
686    3
687    5
688    3
Length: 688, dtype: int64
6152


In [13]:
counts = []

for idx, row in strings_df.iterrows():
    # Calculate valid window
    left_boundary = row['start_x'] + 2
    right_boundary = row['end_x'] - 2

    # Skip if window is invalid
    if left_boundary > right_boundary:
        counts.append(0)
        continue

    # Filter structures
    mask = (
        (mechanical_structures['Position_Y'] == row['Line']) &
        (
            (mechanical_structures['start_x'].between(left_boundary, right_boundary)) |
            (mechanical_structures['end_x'].between(left_boundary, right_boundary)) |
            (mechanical_structures['Position_X'].between(left_boundary, right_boundary))
        )
    )

    count = mechanical_structures[mask].shape[0]
    # Only subtract 1 if count > 0, else keep 0
    counts.append(max(0, count - 1))


strings_df['No of bridges (separates strings)'] = counts
#print(strings_df)
print(mechanical_structures)

       Position_X  Position_Y  No of modules      start_x        end_x  Group
5736 -18354.98299    158305.0             16 -18360.26499 -18349.70099      1
5737 -18344.30899    158305.0             16 -18349.59099 -18339.02699      1
0    -18328.29799    158305.0             32 -18338.87199 -18317.72399      1
5738 -18312.28699    158305.0             16 -18317.56899 -18307.00499      1
1    -18296.27599    158305.0             32 -18306.84999 -18285.70199      1
...           ...         ...            ...          ...          ...    ...
6831 -19069.81791    156362.0             16 -19075.09991 -19064.53591    687
6832 -19112.51391    156354.0             16 -19117.79591 -19107.23191    688
6833 -19101.83991    156354.0             16 -19107.12191 -19096.55791    688
6834 -19091.16591    156354.0             16 -19096.44791 -19085.88391    688
6835 -19080.49191    156354.0             16 -19085.77391 -19075.20991    688

[6840 rows x 6 columns]


In [14]:
print(strings_df['No of bridges (separates strings)'].describe())
print(strings_df['No of bridges (separates strings)'].info())

count    4280.000000
mean        2.205140
std         0.762814
min         0.000000
25%         2.000000
50%         2.000000
75%         3.000000
max         3.000000
Name: No of bridges (separates strings), dtype: float64
<class 'pandas.core.series.Series'>
RangeIndex: 4280 entries, 0 to 4279
Series name: No of bridges (separates strings)
Non-Null Count  Dtype
--------------  -----
4280 non-null   int64
dtypes: int64(1)
memory usage: 33.6 KB
None


In [15]:
bridges_PRK_I = strings_df[strings_df['PRK'] == '1'].copy()
bridges_PRK_II = strings_df[strings_df['PRK'] == '2'].copy()
print(f'There are {bridges_PRK_I[bridges_PRK_I['Type'] != "B"]["No of bridges (separates strings)"].sum()} bridges that separates strings in Capela I and {bridges_PRK_II[bridges_PRK_II['Type'] != "B"]["No of bridges (separates strings)"].sum()} in Capela II.')

There are 2428 bridges that separates strings in Capela I and 2424 in Capela II.


In [16]:
bridges_PRK_I.to_excel(r"C:\Users\Usuario\Desktop\strings_with_bridges_I.xlsx")
bridges_PRK_II.to_excel(r"C:\Users\Usuario\Desktop\strings_with_bridges_II.xlsx")